# Ch 18. 추론 품질 높이기

**[원본 웹페이지](https://desty.github.io/study-ai-assistant-engineering/part4/18-reasoning-quality/)**

## Abstract

- **같은 모델**에서 추론 **전략**을 바꿔 품질을 올리는 4가지 방법
- **CoT** 심화 · **Self-Consistency** · **Best-of-N** · **Verifier**
- **Test-time compute** — 비용을 더 써서 품질을 사는 트레이드오프
- 수학 문제로 self-consistency · 코드 문제로 best-of-N + pytest verifier 구현
- N 을 늘리다가 터지는 비용·지연·verifier 품질 3대 병목

## Dependencies

In [ ]:
!pip install -q anthropic pytest

## API Keys

In [ ]:
import os
from getpass import getpass

if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass('ANTHROPIC_API_KEY: ')

## 1. 개념 — 한 번 묻지 말고, 여러 번 제대로 묻자

LLM 에 같은 질문을 5번 던지면 5개의 다른 답이 나옵니다. 이 **확률성**을 약점이 아니라 **자산**으로 바꾸는 게 이 챕터의 핵심 아이디어입니다.

네 가지 전략:

| 기법 | 아이디어 | 대표 장점 |
|---|---|---|
| ① Single | 직접 답 하나 | 빠르고 싸다 |
| ② **CoT** | "단계별로 생각해" → 추론 trace 유도 | 토큰만 늘리면 됨 |
| ③ **Self-Consistency** | N개 샘플 → 다수결 | 수학·다지선다 QA 에 강력 |
| ④ **Best-of-N + Verifier** | N개 생성 → verifier 가 선택 | 코드·검증 가능 태스크에 최강 |

## 2. 왜 필요한가

**① 모델을 교체할 수 없을 때.** Opus 로 올릴 예산이 없는데 Sonnet 으로 정확도 70% → 85% 가 필요하다.

**② 도메인이 확률적 오답을 감당 못할 때.** 수학·코드·금액 계산은 한 번 틀리면 치명적.

**③ Judge 친화적 세팅이 이미 있을 때.** Ch 17 의 Judge 를 verifier 로 재활용 가능.

## 3. 어디에 쓰이는가 — 기법별 적합 도메인

### 3-1. CoT 심화
- **수학·논리**: "단계별로" 한 줄이 정확도 20~40% 끌어올림
- **복잡한 분류**: 판단 근거를 쓰게 하면 쉬운 패턴 매칭 오류가 줄어듦

### 3-2. Self-Consistency
- **정답이 이산(discrete)**: 숫자 · 카테고리 · 객관식
- 원리: 여러 CoT 경로를 샘플링 → 최종 답만 추출 → **최빈값**

### 3-3. Best-of-N + Verifier
- **검증 가능한 태스크**: 코드(실행) · SQL(DB 검증) · 수학(역대입)
- Verifier 종류: Deterministic / LLM-as-verifier / Reward model

### 3-4. Tree of Thoughts (맥락만)
분기를 트리로 탐색. 구현 복잡도 ↑ · 비용 ↑.

## 4. 최소 예제 — Self-Consistency 로 수학 문제 풀기

In [ ]:
import anthropic
from collections import Counter
import re

client = anthropic.Anthropic()

COT_SYS = """문제를 단계별로 풀고, 마지막 줄에 반드시 다음 형식으로 최종 답을 쓰세요.
ANSWER: <숫자만>"""

def sample_one(question, temperature=0.7):  # (1)!
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=500,
        system=COT_SYS,
        messages=[{'role': 'user', 'content': question}],
    )
    text = msg.content[0].text
    m = re.search(r'ANSWER:\s*([-\d.]+)', text)
    return m.group(1) if m else None

def self_consistency(question, n=5):  # (2)!
    answers = [sample_one(question) for _ in range(n)]
    answers = [a for a in answers if a is not None]
    if not answers:
        return None, 0
    most_common, cnt = Counter(answers).most_common(1)[0]
    return most_common, cnt / n  # 답 + 신뢰도 (일치 비율)

# 예제
q = '한 상자에 24개의 사과가 있고 3명이 똑같이 나눴습니다. 한 명당 몇 개?'
# ans, conf = self_consistency(q, n=5)
# print(f'답: {ans}, 신뢰도: {conf:.0%}')

1. **단일 샘플러** — temperature 를 높여 다양성 확보. CoT 시스템 프롬프트로 추론 유도.
2. **다수결** — 가장 많이 나온 답 + 그 비율(신뢰도). 100% 일치 = 매우 확신. 40% = 경계.

## 5. 실전 튜토리얼 — Best-of-N + pytest verifier (코드 문제)

코드 생성에 **결정적 verifier**(pytest) 를 붙이면 품질이 극적으로 오릅니다.

### 5-1. 구조

```
Q: "두 정수 합 함수 add(a,b) 작성"
  ↓
N=5 generations (temperature=0.8)
  ↓
각 후보를 sandbox 에서 테스트 실행
  ↓
pass 한 후보 중 첫 번째
```

> **Colab 에서 실행 시**: 아래 `subprocess.run(['pytest', ...])` 는 Colab sandbox 에서 `/tmp/` 경로 사용

In [ ]:
import subprocess
import tempfile
import os
import textwrap
import anthropic

client = anthropic.Anthropic()

def generate_candidates(prompt, n=5):
    results = []
    for _ in range(n):
        msg = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=400,
            messages=[{'role': 'user', 'content': prompt}],
        )
        results.append(msg.content[0].text)
    return results

TEST_CODE = """
def test_add():
    from solution import add
    assert add(2, 3) == 5
    assert add(-1, 1) == 0
    assert add(0, 0) == 0
"""

def extract_python(text):
    """마크다운 코드블록에서 python 코드 추출"""
    m = re.search(r'```python\n([\s\S]*?)```', text)
    return m.group(1) if m else text

def verify(code):  # (1)!
    with tempfile.TemporaryDirectory() as td:
        with open(f'{td}/solution.py', 'w') as f:
            f.write(code)
        with open(f'{td}/test_solution.py', 'w') as f:
            f.write(TEST_CODE)
        try:
            r = subprocess.run(
                ['pytest', '-q', td], capture_output=True, text=True, timeout=15
            )
            return r.returncode == 0
        except FileNotFoundError:
            print('pytest not installed in kernel - skipping verification')
            return False
        except subprocess.TimeoutExpired:
            return False

def best_of_n(prompt, n=5):  # (2)!
    for i, cand in enumerate(generate_candidates(prompt, n)):
        code = extract_python(cand)
        if verify(code):
            return code, i
    return None, -1

# 예제
# prompt = '두 정수를 더하는 함수 add(a, b)를 파이썬으로 작성하세요. 반드시 def add(...): 형식으로.'
# code, idx = best_of_n(prompt, n=3)
# if code:
#     print(f'Found working solution at index {idx}')
#     print(code)

1. **Verifier** — pytest 가 결정적으로 pass/fail 판정. LLM 의존이 0.
2. **Best-of-N 루프** — 첫 통과 즉시 반환(조기 종료). 모두 실패면 `None`.

### 5-3. 기대 효과

- N=1: 정확도 60%
- N=5 + verifier: **90%+**  
비용은 평균 ~2~3× (조기 종료 덕분에 5× 안 됨).

## 6. 자주 깨지는 포인트

### 6-1. 토큰·비용 ×N
Self-Consistency N=5 = 비용 × 5 · 지연 × 5(동시 호출 안 하면).

### 6-2. Verifier 가 병목
Best-of-N 품질은 verifier 품질을 **초과할 수 없음**.

### 6-3. 다양성 부족
Temperature 가 0.3 인데 N=20 을 돌리면 거의 같은 답 20개.

### 6-4. 연속 답에 self-consistency
"요약 써주세요" 에 다수결이 안 됨.

### 6-5. Early stopping 함정
Best-of-N 에서 첫 통과 즉시 반환하면 **품질은 동일하지만 평균 비용만** 감소.

## 7. 운영 시 체크할 점

- [ ] Self-Consistency / Best-of-N 을 쓸 **태스크 유형**을 명시했는가 (수학·코드·분류만)
- [ ] N 값을 **평가셋에서 측정**해 결정했는가 (관성적으로 5 쓰지 말 것)
- [ ] Verifier 가 **결정적(pytest·regex)** 인가, **LLM 기반** 인가, 후자라면 합의율 추적하는가
- [ ] 토큰·지연 **×N** 이 운영 SLO 안에 들어오는가
- [ ] 병렬 호출로 지연 상쇄되는가
- [ ] Self-Consistency **신뢰도(일치율)** 를 로깅해, 낮은 건 자동 에스컬레이션 하는가

## 8. 연습문제

### 확인 문제

1. Self-Consistency 가 "요약 태스크" 에 부적합한 이유를 설명하고 대안(best-of-N+judge)이 왜 작동하는지 비교하세요.
2. Best-of-N 에서 verifier 가 LLM Judge 일 때 생기는 추가 리스크 2가지를 드세요.
3. N=1(Single) 과 N=5(Self-Consistency) 의 **기대 비용·지연·품질** 을 수식으로 비교하세요.
4. Test-time compute 를 늘리는 것보다 **모델을 업그레이드**하는 게 나은 경우는 언제인가요?

---

**다음 챕터** → [Ch 19. 실패 분석과 디버깅](../part4/ch19_failure_analysis.ipynb)